# 📊 Applied Data Science with Python — Course 1: Introduction to Data Science

> **Platform:** Coursera | **Institution:** University of Michigan  
> **Level:** Intermediate | **Language:** Python 3

---

## 📋 About This Course

This course introduces data science through Python, focusing on data manipulation with pandas and numpy, statistical analysis, and real-world dataset exploration. Assignments progress from regex-based log parsing to complex multi-dataset merges and hypothesis testing.

---

## 📚 Table of Contents

| Week | Topic | Key Skills |
|------|-------|------------|
| Week 1 | Regex & Log Parsing | `re`, named groups, `finditer` |
| Week 2 | Pandas & CDC Immunization Data | `pd.read_csv`, groupby, correlation |
| Week 3 | Energy, GDP & World Bank Data | `pd.read_excel`, merge, advanced groupby |
| Week 4 | Sports Data & Hypothesis Testing | `scipy.stats`, win/loss ratios, ttest |

---

## 📌 Week 1 — Regex & Log Parsing

> **Dataset:** `logdata.txt` — Apache web server access log  
> **Goal:** Use regular expressions with named groups to parse structured log data.

---

# Assignment 1
For this assignment you are welcomed to use other regex resources such a regex "cheat sheets" you find on the web.



Before start working on the problems, here is a small example to help you understand how to write your own answers. In short, the solution should be written within the function body given, and the final result should be returned. Then the autograder will try to call the function and validate your returned result accordingly. 

In [13]:
def example_word_count():
    # This example question requires counting words in the example_string below.
    example_string = "Amy is 5 years old"
    
    # YOUR CODE HERE.
    # You should write your solution here, and return your result, you can comment out or delete the
    # NotImplementedError below.
    result = example_string.split(" ")
    return len(result)

    #raise NotImplementedError()

## Part A

Find a list of all of the names in the following string using regex.

In [273]:
import re
def names():
    simple_string = """Amy is 5 years old, and her sister Mary is 2 years old. 
    Ruth and Peter, their parents, have 3 kids."""

    # YOUR CODE HERE
    pattern="[A-Z][a-z]*"  
    names=re.findall(pattern,simple_string)
    print(names)
    return names

In [274]:
assert len(names()) == 4, "There are four names in the simple_string"


['Amy', 'Mary', 'Ruth', 'Peter']


## Part B

The dataset file in [assets/grades.txt](assets/grades.txt) contains a line separated list of people with their grade in 
a class. Create a regex to generate a list of just those students who received a B in the course.

In [374]:
import re
def grades():
    with open ("assets/grades.txt", "r") as file:
        grades = file.read()
        pattern="(\w+\s\w+): B"
        grades=re.findall(pattern,grades)
        print(grades)
    return grades


In [375]:
assert len(grades()) == 16


['Bell Kassulke', 'Simon Loidl', 'Elias Jovanovic', 'Hakim Botros', 'Emilie Lorentsen', 'Jake Wood', 'Fatemeh Akhtar', 'Kim Weston', 'Yasmin Dar', 'Viswamitra Upandhye', 'Killian Kaufman', 'Elwood Page', 'Elodie Booker', 'Adnan Chen', 'Hank Spinka', 'Hannah Bayer']


## Part C

Consider the standard web log file in [assets/logdata.txt](assets/logdata.txt). This file records the access a user makes when visiting a web page (like this one!). Each line of the log has the following items:
* a host (e.g., '146.204.224.152') 
* a user_name (e.g., 'feest6811' **note: sometimes the user name is missing! In this case, use '-' as the value for the username.**)
* the time a request was made (e.g., '21/Jun/2019:15:45:24 -0700')
* the post request type (e.g., 'POST /incentivize HTTP/1.1' **note: not everything is a POST!**)

Your task is to convert this into a list of dictionaries, where each dictionary looks like the following:
```
example_dict = {"host":"146.204.224.152", 
                "user_name":"feest6811", 
                "time":"21/Jun/2019:15:45:24 -0700",
                "request":"POST /incentivize HTTP/1.1"}
```

In [1]:
import re
def logs():
    with open("assets/logdata.txt", "r") as file:
        logdata = file.read()
        pattern="""(?P<host>.*)                #hostname
                   (\s-\s)                     #formato de lo que sigue
                   (?P<user_name>.*.(?=\s\[))  #grupo username es:todos los caracteres que coincidan 0 o + veces (.*) seguido
                                               #de todos los caracteres hasta que encuentre espacio y corchete .(?=\s\[)
                   (\s\[)                      #formato de lo que sigue 
                   (?P<time>.*(?=\]))          #grupo time es: todos los caracteres hasta que encuentre [                    
                   (\]\s\")                    #formato de lo que sigue
                   (?P<request>.*(?=\"))       #grupo request es: todos los caracteres hasta que encuentre " que cierran     
                   
                """
        logs=[]
        for item in re.finditer(pattern,logdata,re.VERBOSE):
            a=item.groupdict()
            logs.append(a)
            print(logs)
            
        print(len(logs))    
    return logs
    

In [2]:
assert len(logs()) == 979

one_item={'host': '146.204.224.152',
  'user_name': 'feest6811',
  'time': '21/Jun/2019:15:45:24 -0700',
  'request': 'POST /incentivize HTTP/1.1'}
assert one_item in logs(), "Sorry, this item should be in the log results, check your formating"


IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



## 📌 Week 2 — Pandas & CDC Immunization Data

> **Dataset:** `NISPUF17.csv` — CDC 2017 National Immunization Survey  
> **Goal:** Explore relationships between education, breastfeeding, vaccines and chickenpox using pandas.

---

# Assignment 2
For this assignment you'll be looking at 2017 data on immunizations from the CDC. Your datafile for this assignment is in [assets/NISPUF17.csv](assets/NISPUF17.csv). A data users guide for this, which you'll need to map the variables in the data to the questions being asked, is available at [assets/NIS-PUF17-DUG.pdf](assets/NIS-PUF17-DUG.pdf). **Note: you may have to go to your Jupyter tree (click on the Coursera image) and navigate to the assignment 2 assets folder to see this PDF file).**

## Question 1
Write a function called `proportion_of_education` which returns the proportion of children in the dataset who had a mother with the education levels equal to less than high school (<12), high school (12), more than high school but not a college graduate (>12) and college degree.

*This function should return a dictionary in the form of (use the correct numbers, do not round numbers):* 
```
    {"less than high school":0.2,
    "high school":0.4,
    "more than high school but not college":0.2,
    "college":0.2}
```


In [18]:
import pandas as pd
df= pd.read_csv('assets/NISPUF17.csv', index_col=0)
ed=df['EDUC1']

def proportion_of_education():
    
    NumMothers=len(ed)
    lessth=[]
    eq=[]
    higherth=[]
    colleg=[]
    for i in ed:
        if   i == 1:
            lessth.append(i)
            portionless=len(lessth)/NumMothers
        elif i == 2:
            eq.append(i)
            portioneq=len(eq)/NumMothers
        elif i == 3:
            higherth.append(i)
            portionhigher=len(higherth)/NumMothers
        else :
            colleg.append(i)
            portioncolleg=len(colleg)/NumMothers
    proportion_of_education= {"less than high school":portionless,
                              "high school":portioneq,
                              "more than high school but not college":portionhigher,
                              "college":portioncolleg}
    return proportion_of_education


In [19]:
assert type(proportion_of_education())==type({}), "You must return a dictionary."
assert len(proportion_of_education()) == 4, "You have not returned a dictionary with four items in it."
assert "less than high school" in proportion_of_education().keys(), "You have not returned a dictionary with the correct keys."
assert "high school" in proportion_of_education().keys(), "You have not returned a dictionary with the correct keys."
assert "more than high school but not college" in proportion_of_education().keys(), "You have not returned a dictionary with the correct keys."
assert "college" in proportion_of_education().keys(), "You have not returned a dictionary with the correct keys."


## Question 2

Let's explore the relationship between being fed breastmilk as a child and getting a seasonal influenza vaccine from a healthcare provider. Return a tuple of the average number of influenza vaccines for those children we know received breastmilk as a child and those who know did not.

*This function should return a tuple in the form (use the correct numbers:*
```
(2.5, 0.1)
```

In [4]:
import pandas as pd
df= pd.read_csv('assets/NISPUF17.csv', index_col=0)

def average_influenza_doses():
    copia= df.drop(df[df['CBF_01']>2].index)       #aplico filtro 1. CBF=1 o 2.
    columnas=copia[["CBF_01","P_NUMFLU"]].dropna() #aplico filtro2. P_NUMFLU sin datos Nan
    fedyes=columnas[columnas['CBF_01']==1]         #con esto saco valores de P_NUMFLU cuando CBF=1
    promfedyes=fedyes['P_NUMFLU'].mean()           #promedio valores de P_NUMFLU cuando CBF=1
    fedNo=columnas[columnas['CBF_01']==2]          #con esto saco valores de P_NUMFLU cuando CBF=2
    promfedNo=fedNo['P_NUMFLU'].mean()             #promedio valores de P_NUMFLU cuando CBF=2
    average_influenza_doses=(promfedyes,promfedNo)
    return average_influenza_doses


In [5]:
assert len(average_influenza_doses())==2, "Return two values in a tuple, the first for yes and the second for no."


## Question 3
It would be interesting to see if there is any evidence of a link between vaccine effectiveness and sex of the child. Calculate the ratio of the number of children who contracted chickenpox but were vaccinated against it (at least one varicella dose) versus those who were vaccinated but did not contract chicken pox. Return results by sex. 

*This function should return a dictionary in the form of (use the correct numbers):* 
```
    {"male":0.2,
    "female":0.4}
```

Note: To aid in verification, the `chickenpox_by_sex()['female']` value the autograder is looking for starts with the digits `0.0077`.

In [8]:
import pandas as pd
df=pd.read_csv('assets/NISPUF17.csv', index_col=0)
def chickenpox_by_sex():
    copy= df
    columnas=copy[["HAD_CPOX","P_NUMVRC", "SEX"]]
    print(columnas)
    print('\n')
    #------------------------------- male -------------------------------------
    mask1=(columnas['SEX']==1) & (columnas['HAD_CPOX']==1) & (columnas['P_NUMVRC']>=1)
    hvc=columnas.where(mask1).dropna()            #Dataframe de hombres con vacuna contagiados
    print(hvc)
    thc=len(hvc)                                  #total hombres contagiados con al menos una vacuna
    print('Hombres contagiados con al menos una vacuna : ', thc)

    mask2=(columnas['SEX']==1) & (columnas['HAD_CPOX']==2) & (columnas['P_NUMVRC']>=1)
    hvnc=columnas.where(mask2).dropna()           #Dataframe de hombres con vacuna no contagiados
    print('\n',hvnc)
    thnc=len(hvnc)                                #total hombres no contagiados con al menos una vacuna
    print('Hombres no contagiados con al menos una vacuna : ', thnc)

    maleratio=thc/thnc                            #Hombres contagiados vs No contagiados, todos con vacuna
    print('\nMale Ratio', maleratio)
    #------------------------------- female -----------------------------------
    mask3=(columnas['SEX']==2) & (columnas['HAD_CPOX']==1) & (columnas['P_NUMVRC']>=1)
    mvc=columnas.where(mask3).dropna()            #Dataframe de mujeres con vacuna contagiadas
    print('\n',mvc)
    tmc=len(mvc)                                  #total mujeres contagiadass con al menos una vacuna
    print('\nMujeres contagiadas con al menos una vacuna : ', tmc)

    mask4=(columnas['SEX']==2) & (columnas['HAD_CPOX']==2) & (columnas['P_NUMVRC']>=1)
    mvnc=columnas.where(mask4).dropna()           #Dataframe de mujeres con vacuna no contagiadas
    print('\n',mvnc)
    tmnc=len(mvnc)                                #total mujeres no contagiados con al menos una vacuna
    print('\nMujeres no contagiadas con al menos una vacuna : ', tmnc)

    femaleratio=tmc/tmnc                            #Hombres contagiados vs No contagiados, todos con vacuna
    print('\nFemale Ratio', femaleratio)
    
    chickenpox_by_sex={"male":maleratio, "female":femaleratio}
    print('\n',chickenpox_by_sex)
    return chickenpox_by_sex



In [9]:
assert len(chickenpox_by_sex())==2, "Return a dictionary with two items, the first for males and the second for females."


       HAD_CPOX  P_NUMVRC  SEX
1             2       NaN    1
2             2       NaN    1
3             2       NaN    2
4             2       1.0    2
5             2       0.0    2
...         ...       ...  ...
28461         2       NaN    2
28462         2       NaN    2
28463         2       NaN    2
28464         2       NaN    2
28465         2       NaN    2

[28465 rows x 3 columns]


       HAD_CPOX  P_NUMVRC  SEX
462         1.0       1.0  1.0
688         1.0       1.0  1.0
864         1.0       1.0  1.0
1480        1.0       1.0  1.0
1907        1.0       1.0  1.0
...         ...       ...  ...
27046       1.0       1.0  1.0
27721       1.0       1.0  1.0
28107       1.0       1.0  1.0
28108       1.0       1.0  1.0
28405       1.0       1.0  1.0

[68 rows x 3 columns]
Hombres contagiados con al menos una vacuna :  68

        HAD_CPOX  P_NUMVRC  SEX
11          2.0       1.0  1.0
13          2.0       1.0  1.0
17          2.0       2.0  1.0
20          2.0       1.0  1.

## Question 4
A correlation is a statistical relationship between two variables. If we wanted to know if vaccines work, we might look at the correlation between the use of the vaccine and whether it results in prevention of the infection or disease [1]. In this question, you are to see if there is a correlation between having had the chicken pox and the number of chickenpox vaccine doses given (varicella).

Some notes on interpreting the answer. The `had_chickenpox_column` is either `1` (for yes) or `2` (for no), and the `num_chickenpox_vaccine_column` is the number of doses a child has been given of the varicella vaccine. A positive correlation (e.g., `corr > 0`) means that an increase in `had_chickenpox_column` (which means more no’s) would also increase the values of `num_chickenpox_vaccine_column` (which means more doses of vaccine). If there is a negative correlation (e.g., `corr < 0`), it indicates that having had chickenpox is related to an increase in the number of vaccine doses.

Also, `pval` is the probability that we observe a correlation between `had_chickenpox_column` and `num_chickenpox_vaccine_column` which is greater than or equal to a particular value occurred by chance. A small `pval` means that the observed correlation is highly unlikely to occur by chance. In this case, `pval` should be very small (will end in `e-18` indicating a very small number).

[1] This isn’t really the full picture, since we are not looking at when the dose was given. It’s possible that children had chickenpox and then their parents went to get them the vaccine. Does this dataset have the data we would need to investigate the timing of the dose?

In [30]:
import scipy.stats as stats
import pandas as pd
df=pd.read_csv('assets/NISPUF17.csv', index_col=0)
def corr_chickenpox():
    copy= df
    columnas=copy[["HAD_CPOX","P_NUMVRC"]]
    mask4=(columnas["HAD_CPOX"].lt(3))
    res=columnas.where(mask4).dropna()
    res.sort_index(inplace=True)
    #print(res)

    #print(res["HAD_CPOX"].unique())
    #print(res["P_NUMVRC"].unique())


    # here is some stub code to actually run the correlation
    corr, pval=stats.pearsonr(res["HAD_CPOX"],res["P_NUMVRC"])
    #print('Correlacion: ',corr)
    #print('pval:         ', pval)
    return corr
corr_chickenpox()

   

0.07044873460147986

In [31]:
assert -1<=corr_chickenpox()<=1, "You must return a float number between -1.0 and 1.0."


## 📌 Week 3 — Energy, GDP & World Bank Data

> **Datasets:** `Energy_Indicators.xls`, `world_bank.csv`, `scimagojr-3.xlsx`  
> **Goal:** Merge 3 real-world datasets and answer 13 analytical questions about global energy and economic indicators.

---

# Assignment 3
All questions are weighted the same in this assignment. This assignment requires more individual learning then the last one did - you are encouraged to check out the [pandas documentation](http://pandas.pydata.org/pandas-docs/stable/) to find functions or methods you might not have used yet, or ask questions on [Stack Overflow](http://stackoverflow.com/) and tag them as pandas and python related. All questions are worth the same number of points except question 1 which is worth 17% of the assignment grade.

**Note**: Questions 2-13 rely on your question 1 answer.

In [2]:

# Filter all warnings. If you would like to see the warnings, please comment the two lines below.
#import warnings
#warnings.filterwarnings('ignore')

### Question 1
Load the energy data from the file `assets/Energy Indicators.xls`, which is a list of indicators of [energy supply and renewable electricity production](assets/Energy%20Indicators.xls) from the [United Nations](http://unstats.un.org/unsd/environment/excel_file_tables/2013/Energy%20Indicators.xls) for the year 2013, and should be put into a DataFrame with the variable name of **Energy**.

Keep in mind that this is an Excel file, and not a comma separated values file. Also, make sure to exclude the footer and header information from the datafile. The first two columns are unneccessary, so you should get rid of them, and you should change the column labels so that the columns are:

`['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable]`

Convert `Energy Supply` to gigajoules (**Note: there are 1,000,000 gigajoules in a petajoule**). For all countries which have missing data (e.g. data with "...") make sure this is reflected as `np.NaN` values.

Rename the following list of countries (for use in later questions):

```"Republic of Korea": "South Korea",
"United States of America": "United States",
"United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
"China, Hong Kong Special Administrative Region": "Hong Kong"```

There are also several countries with numbers and/or parenthesis in their name. Be sure to remove these, e.g. `'Bolivia (Plurinational State of)'` should be `'Bolivia'`.  `'Switzerland17'` should be `'Switzerland'`.

Next, load the GDP data from the file `assets/world_bank.csv`, which is a csv containing countries' GDP from 1960 to 2015 from [World Bank](http://data.worldbank.org/indicator/NY.GDP.MKTP.CD). Call this DataFrame **GDP**. 

Make sure to skip the header, and rename the following list of countries:

```"Korea, Rep.": "South Korea", 
"Iran, Islamic Rep.": "Iran",
"Hong Kong SAR, China": "Hong Kong"```

Finally, load the [Sciamgo Journal and Country Rank data for Energy Engineering and Power Technology](http://www.scimagojr.com/countryrank.php?category=2102) from the file `assets/scimagojr-3.xlsx`, which ranks countries based on their journal contributions in the aforementioned area. Call this DataFrame **ScimEn**.

Join the three datasets: GDP, Energy, and ScimEn into a new dataset (using the intersection of country names). Use only the last 10 years (2006-2015) of GDP data and only the top 15 countries by Scimagojr 'Rank' (Rank 1 through 15). 

The index of this DataFrame should be the name of the country, and the columns should be ['Rank', 'Documents', 'Citable documents', 'Citations', 'Self-citations',
       'Citations per document', 'H index', 'Energy Supply',
       'Energy Supply per Capita', '% Renewable', '2006', '2007', '2008',
       '2009', '2010', '2011', '2012', '2013', '2014', '2015'].

*This function should return a DataFrame with 20 columns and 15 entries, and the rows of the DataFrame should be sorted by "Rank".*

In [3]:
import numpy as np
import re
import pandas as pd

def answer_one():
    Energy= pd.read_excel("assets/Energy Indicators.xls", sheet_name = "Energy")
    Energy=Energy.drop(Energy.columns[[0, 1]], axis='columns')  #elimino columnas por su indice 0 y 1.
    Energy=Energy.drop(range(0,17,1), axis=0)                   #eliino rango de filas 0 a 17 (header)
    Energy=Energy.drop(range(244,282,1), axis=0)                #eliino rango de filas 244 a 282 (footer)
    Energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'] # Renombro columnas
    listaener=[]
    for k in Energy.columns:
        str(k).strip()
        listaener.append(k)
    Energy.set_axis(listaener, axis=1, inplace=True)
    Energy[Energy == '...'] = np.nan                            #relleno con NaN donde haya puntos (...)
    Energy['Country'].replace('[\d*]','', regex=True, inplace=True)  #reemplazar con vacio '' donde cadena de numeros
    Energy['Country'].replace('\(.*\)','', regex=True, inplace=True) #reemplazar con vacio '' donde cadena de numeros ()
    Energy['Country']=Energy['Country'].str.rstrip()            #elimino espacios que sobran al fin de la cadena
    Energy['Country'].replace({'Republic of Korea': 'South Korea', 'United States of America': 'United States',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'China, Hong Kong Special Administrative Region': 'Hong Kong'}, inplace=True) #reemplazo nombres
    Energy['Energy Supply']=Energy['Energy Supply']*1000000     #paso de peta a giga joules
    Energy=Energy.set_index('Country')
    # print(Energy)
    # print(len(Energy)) #227

    GDP= pd.read_csv('assets/world_bank.csv', header=None, skiprows=4) #elimino header y las 4 priemras filas
    GDP[0]=GDP[0].str.strip()                                   #remuevo espacios de la columna 0
    axislist=[]
    for x in GDP.iloc[0]:                                       #creo lista con los nombres de las cabeceras
        axislist.append(x)
    for k in range (4,len(axislist),1):
        axislist[k]=str(int(axislist[k]))
    GDP.drop([0],axis=0,inplace=True)                           #elimino la fila 0
    GDP.set_axis(axislist, axis=1, inplace=True)                #coloco lista como nuevo axis
    GDP.rename(columns={'Country Name':'Country'}, inplace=True)#cambio nombre de cabecera
    GDP['Country'].replace({"Korea, Rep." : "South Korea", "Iran, Islamic Rep.": "Iran", "Hong Kong SAR, China": "Hong Kong"}, inplace=True)
    GDP=GDP.set_index('Country')
    # print(GDP)
    # print(len(GDP)) #264

    ScimEn= pd.read_excel("assets/scimagojr-3.xlsx", sheet_name = "Sheet1")
    ScimEn=ScimEn.set_index('Country')
    # print(ScimEn)
    # print(len(ScimEn)) #191

    mergedf=pd.merge(ScimEn, Energy, how='outer', left_index=True, right_index=True)  #uno GDP Y Energy dataframes en un solo dataf: mergedf
    df=pd.merge(mergedf, GDP, how='outer', left_index=True, right_index=True)   #uno mergedf con Scim dataframe= todos dataframes en una sola= df
    df=df.drop(df.columns[10:59], axis='columns') #elimino columnas que no necesito
    df.sort_values(by=['Rank'], inplace=True)
    df=df.iloc[0:15]
    lista=[]
    for x in df.columns:
        str(x).strip()
        lista.append(x)
    df.set_axis(lista, axis=1, inplace=True)
    answer_one=df                          #filtrada por los primeros 15 valores = (15 valores de rank)
    #print(type(answer_one))
    #print(answer_one.shape)
    #print(answer_one)
    return(answer_one)

In [4]:
assert type(answer_one()) == pd.DataFrame, "Q1: You should return a DataFrame!"

assert answer_one().shape == (15,20), "Q1: Your DataFrame should have 20 columns and 15 entries!"


In [5]:
# Cell for autograder.


### Question 2
The previous question joined three datasets then reduced this to just the top 15 entries. When you joined the datasets, but before you reduced this to the top 15 items, how many entries did you lose?

*This function should return a single number.*

In [6]:
%%HTML
<svg width="800" height="300">
  <circle cx="150" cy="180" r="80" fill-opacity="0.2" stroke="black" stroke-width="2" fill="blue" />
  <circle cx="200" cy="100" r="80" fill-opacity="0.2" stroke="black" stroke-width="2" fill="red" />
  <circle cx="100" cy="100" r="80" fill-opacity="0.2" stroke="black" stroke-width="2" fill="green" />
  <line x1="150" y1="125" x2="300" y2="150" stroke="black" stroke-width="2" fill="black" stroke-dasharray="5,3"/>
  <text x="300" y="165" font-family="Verdana" font-size="35">Everything but this!</text>
</svg>

In [10]:
import numpy as np
import re
import pandas as pd
def answer_two():
    Energy= pd.read_excel("assets/Energy Indicators.xls", sheet_name = "Energy")
    Energy=Energy.drop(Energy.columns[[0, 1]], axis='columns')  #elimino columnas por su indice 0 y 1.
    Energy=Energy.drop(range(0,17,1), axis=0)                   #eliino rango de filas 0 a 17 (header)
    Energy=Energy.drop(range(244,282,1), axis=0)                #eliino rango de filas 244 a 282 (footer)
    Energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'] # Renombro columnas
    listaener=[]
    for k in Energy.columns:
        str(k).strip()
        listaener.append(k)
    Energy.set_axis(listaener, axis=1, inplace=True)
    Energy[Energy == '...'] = np.nan                            #relleno con NaN donde haya puntos (...)
    Energy['Country'].replace('[\d*]','', regex=True, inplace=True)  #reemplazar con vacio '' donde cadena de numeros
    Energy['Country'].replace('\(.*\)','', regex=True, inplace=True) #reemplazar con vacio '' donde cadena de numeros ()
    Energy['Country']=Energy['Country'].str.rstrip()            #elimino espacios que sobran al fin de la cadena
    Energy['Country'].replace({'Republic of Korea': 'South Korea', 'United States of America': 'United States',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'China, Hong Kong Special Administrative Region': 'Hong Kong'}, inplace=True) #reemplazo nombres
    Energy['Energy Supply']=Energy['Energy Supply']*1000000     #paso de peta a giga joules
    Energy=Energy.set_index('Country')
    # print(Energy)
    # print(len(Energy)) #227

    GDP= pd.read_csv('assets/world_bank.csv', header=None, skiprows=4) #elimino header y las 4 priemras filas
    GDP[0]=GDP[0].str.strip()                                   #remuevo espacios de la columna 0
    axislist=[]
    for x in GDP.iloc[0]:                                       #creo lista con los nombres de las cabeceras
        axislist.append(x)
    for k in range (4,len(axislist),1):
        axislist[k]=str(int(axislist[k]))
    GDP.drop([0],axis=0,inplace=True)                           #elimino la fila 0
    GDP.set_axis(axislist, axis=1, inplace=True)                #coloco lista como nuevo axis
    GDP.rename(columns={'Country Name':'Country'}, inplace=True)#cambio nombre de cabecera
    GDP['Country'].replace({"Korea, Rep." : "South Korea", "Iran, Islamic Rep.": "Iran", "Hong Kong SAR, China": "Hong Kong"}, inplace=True)
    GDP=GDP.set_index('Country')
    # print(GDP)
    # print(len(GDP)) #264

    ScimEn= pd.read_excel("assets/scimagojr-3.xlsx", sheet_name = "Sheet1")
    ScimEn['Country']=ScimEn['Country'].str.rstrip()
    ScimEn=ScimEn.set_index('Country')
    # print(ScimEn)
    # print(len(ScimEn)) #191

   
    mergedf=pd.merge(ScimEn, Energy, how='outer', left_index=True, right_index=True)  #uno GDP Y Energy dataframes en un solo dataf: mergedf
    df=pd.merge(mergedf, GDP, how='outer', left_index=True, right_index=True)   #uno mergedf con Scim dataframe= todos dataframes en una sola= df
    outervalues=len(df)
    print('Outervalues length: ', outervalues)
    
    mergedf1=pd.merge(ScimEn, Energy, on="Country")  #uno GDP Y Energy dataframes en un solo dataf: mergedf
    df1=pd.merge(mergedf1, GDP, on="Country")   #uno mergedf con Scim dataframe= todos dataframe
    innervalues=len(df1)
    print('innervalues length: ',innervalues)
    
    
    answer_two=outervalues-innervalues
    print('lost Values: ', answer_two)
    
    return(answer_two)
    



   


In [11]:
assert type(answer_two()) == int, "Q2: You should return an int number!"


Outervalues length:  318
innervalues length:  162
lost Values:  156


### Question 3
What are the top 15 countries for average GDP over the last 10 years?

*This function should return a Series named `avgGDP` with 15 countries and their average GDP sorted in descending order.*

In [32]:
import numpy as np
import re
import pandas as pd
def answer_three():
    Energy= pd.read_excel("assets/Energy Indicators.xls", sheet_name = "Energy")
    Energy=Energy.drop(Energy.columns[[0, 1]], axis='columns')  #elimino columnas por su indice 0 y 1.
    Energy=Energy.drop(range(0,17,1), axis=0)                   #eliino rango de filas 0 a 17 (header)
    Energy=Energy.drop(range(244,282,1), axis=0)                #eliino rango de filas 244 a 282 (footer)
    Energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'] # Renombro columnas
    listaener=[]
    for k in Energy.columns:
        str(k).strip()
        listaener.append(k)
    Energy.set_axis(listaener, axis=1, inplace=True)
    Energy[Energy == '...'] = np.nan                            #relleno con NaN donde haya puntos (...)
    Energy['Country'].replace('[\d*]','', regex=True, inplace=True)  #reemplazar con vacio '' donde cadena de numeros
    Energy['Country'].replace('\(.*\)','', regex=True, inplace=True) #reemplazar con vacio '' donde cadena de numeros ()
    Energy['Country']=Energy['Country'].str.rstrip()            #elimino espacios que sobran al fin de la cadena
    Energy['Country'].replace({'Republic of Korea': 'South Korea', 'United States of America': 'United States',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'China, Hong Kong Special Administrative Region': 'Hong Kong'}, inplace=True) #reemplazo nombres
    Energy['Energy Supply']=Energy['Energy Supply']*1000000     #paso de peta a giga joules
    Energy=Energy.set_index('Country')
    # print(Energy)
    # print(len(Energy)) #227

    GDP= pd.read_csv('assets/world_bank.csv', header=None, skiprows=4) #elimino header y las 4 priemras filas
    GDP[0]=GDP[0].str.strip()                                   #remuevo espacios de la columna 0
    axislist=[]
    for x in GDP.iloc[0]:                                       #creo lista con los nombres de las cabeceras
        axislist.append(x)
    for k in range (4,len(axislist),1):
        axislist[k]=str(int(axislist[k]))
    GDP.drop([0],axis=0,inplace=True)                           #elimino la fila 0
    GDP.set_axis(axislist, axis=1, inplace=True)                #coloco lista como nuevo axis
    GDP.rename(columns={'Country Name':'Country'}, inplace=True)#cambio nombre de cabecera
    GDP['Country'].replace({"Korea, Rep." : "South Korea", "Iran, Islamic Rep.": "Iran", "Hong Kong SAR, China": "Hong Kong"}, inplace=True)
    GDP=GDP.set_index('Country')
    # print(GDP)
    # print(len(GDP)) #264

    ScimEn= pd.read_excel("assets/scimagojr-3.xlsx", sheet_name = "Sheet1")
    ScimEn=ScimEn.set_index('Country')
    # print(ScimEn)
    # print(len(ScimEn)) #191

    mergedf=pd.merge(ScimEn, Energy, how='outer', left_index=True, right_index=True)  #uno GDP Y Energy dataframes en un solo dataf: mergedf
    df=pd.merge(mergedf, GDP, how='outer', left_index=True, right_index=True)   #uno mergedf con Scim dataframe= todos dataframes en una sola= df
    df=df.drop(df.columns[10:59], axis='columns') #elimino columnas que no necesito
    df.sort_values(by=['Rank'], inplace=True)
    df=df.iloc[0:15]
    
    lista=[]
    for x in df.columns:                       #quita espacios de las columnas
        str(x).strip()
        lista.append(x)
    df.set_axis(lista, axis=1, inplace=True)  #reestablece los nuevos nombres de columnas

    df['avgGDP']=df.iloc[:,10:].mean(axis=1)
    df=df['avgGDP'][0:15]
    df.sort_values(ascending=False, inplace=True)
    
    answer_three=df   #filtrada por los primeros 15 valores = (15 valores de rank)
    
    print(answer_three)
    print(type(answer_three))
    
    return(answer_three)


In [33]:
assert type(answer_three()) == pd.Series, "Q3: You should return a Series!"


Country
United States         1.536434e+13
China                 6.348609e+12
Japan                 5.542208e+12
Germany               3.493025e+12
France                2.681725e+12
United Kingdom        2.487907e+12
Brazil                2.189794e+12
Italy                 2.120175e+12
India                 1.769297e+12
Canada                1.660647e+12
Russian Federation    1.565459e+12
Spain                 1.418078e+12
Australia             1.164043e+12
South Korea           1.106715e+12
Iran                  4.441558e+11
Name: avgGDP, dtype: float64
<class 'pandas.core.series.Series'>


### Question 4
By how much had the GDP changed over the 10 year span for the country with the 6th largest average GDP?

*This function should return a single number.*

In [5]:
import numpy as np
import re
import pandas as pd
def answer_four():
    Energy= pd.read_excel("assets/Energy Indicators.xls", sheet_name = "Energy")
    Energy=Energy.drop(Energy.columns[[0, 1]], axis='columns')  #elimino columnas por su indice 0 y 1.
    Energy=Energy.drop(range(0,17,1), axis=0)                   #eliino rango de filas 0 a 17 (header)
    Energy=Energy.drop(range(244,282,1), axis=0)                #eliino rango de filas 244 a 282 (footer)
    Energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'] # Renombro columnas
    listaener=[]
    for k in Energy.columns:
        str(k).strip()
        listaener.append(k)
    Energy.set_axis(listaener, axis=1, inplace=True)
    Energy[Energy == '...'] = np.nan                            #relleno con NaN donde haya puntos (...)
    Energy['Country'].replace('[\d*]','', regex=True, inplace=True)  #reemplazar con vacio '' donde cadena de numeros
    Energy['Country'].replace('\(.*\)','', regex=True, inplace=True) #reemplazar con vacio '' donde cadena de numeros ()
    Energy['Country']=Energy['Country'].str.rstrip()            #elimino espacios que sobran al fin de la cadena
    Energy['Country'].replace({'Republic of Korea': 'South Korea', 'United States of America': 'United States',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'China, Hong Kong Special Administrative Region': 'Hong Kong'}, inplace=True) #reemplazo nombres
    Energy['Energy Supply']=Energy['Energy Supply']*1000000     #paso de peta a giga joules
    Energy=Energy.set_index('Country')
    # print(Energy)
    # print(len(Energy)) #227

    GDP= pd.read_csv('assets/world_bank.csv', header=None, skiprows=4) #elimino header y las 4 priemras filas
    GDP[0]=GDP[0].str.strip()                                   #remuevo espacios de la columna 0
    axislist=[]
    for x in GDP.iloc[0]:                                       #creo lista con los nombres de las cabeceras
        axislist.append(x)
    for k in range (4,len(axislist),1):
        axislist[k]=str(int(axislist[k]))
    GDP.drop([0],axis=0,inplace=True)                           #elimino la fila 0
    GDP.set_axis(axislist, axis=1, inplace=True)                #coloco lista como nuevo axis
    GDP.rename(columns={'Country Name':'Country'}, inplace=True)#cambio nombre de cabecera
    GDP['Country'].replace({"Korea, Rep." : "South Korea", "Iran, Islamic Rep.": "Iran", "Hong Kong SAR, China": "Hong Kong"}, inplace=True)
    GDP=GDP.set_index('Country')
    # print(GDP)
    # print(len(GDP)) #264

    ScimEn= pd.read_excel("assets/scimagojr-3.xlsx", sheet_name = "Sheet1")
    ScimEn=ScimEn.set_index('Country')
    # print(ScimEn)
    # print(len(ScimEn)) #191

    mergedf=pd.merge(ScimEn, Energy, how='outer', left_index=True, right_index=True)  #uno GDP Y Energy dataframes en un solo dataf: mergedf
    df=pd.merge(mergedf, GDP, how='outer', left_index=True, right_index=True)   #uno mergedf con Scim dataframe= todos dataframes en una sola= df
    df=df.drop(df.columns[10:59], axis='columns') #elimino columnas que no necesito
    df.sort_values(by=['Rank'], inplace=True)
    df=df.iloc[0:15]
    
    lista=[]
    for x in df.columns:                       #quita espacios de las columnas
        str(x).strip()
        lista.append(x)
    df.set_axis(lista, axis=1, inplace=True)  #reestablece los nuevos nombres de columnas
    df['avgGDP']=df.iloc[:,10:].mean(axis=1)
    #df.sort_values(ascending=False, inplace=True) descending average GDP Countries
    print(df)
    # print(df.columns)
    answer_four=df.iloc[3,19] - df.iloc[3,10] #6th largest average GDP:unitedkindome:2015 - 2006 =
    # print(answer_four)

    print(answer_four)
    print(type(answer_four))

    return(answer_four)


In [6]:
# Cell for autograder.


### Question 5
What is the mean energy supply per capita?

*This function should return a single number.*

In [13]:
import numpy as np
import re
import pandas as pd
def answer_five():
    Energy= pd.read_excel("assets/Energy Indicators.xls", sheet_name = "Energy")
    Energy=Energy.drop(Energy.columns[[0, 1]], axis='columns')  #elimino columnas por su indice 0 y 1.
    Energy=Energy.drop(range(0,17,1), axis=0)                   #eliino rango de filas 0 a 17 (header)
    Energy=Energy.drop(range(244,282,1), axis=0)                #eliino rango de filas 244 a 282 (footer)
    Energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'] # Renombro columnas
    listaener=[]
    for k in Energy.columns:
        str(k).strip()
        listaener.append(k)
    Energy.set_axis(listaener, axis=1, inplace=True)
    Energy[Energy == '...'] = np.nan                            #relleno con NaN donde haya puntos (...)
    Energy['Country'].replace('[\d*]','', regex=True, inplace=True)  #reemplazar con vacio '' donde cadena de numeros
    Energy['Country'].replace('\(.*\)','', regex=True, inplace=True) #reemplazar con vacio '' donde cadena de numeros ()
    Energy['Country']=Energy['Country'].str.rstrip()            #elimino espacios que sobran al fin de la cadena
    Energy['Country'].replace({'Republic of Korea': 'South Korea', 'United States of America': 'United States',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'China, Hong Kong Special Administrative Region': 'Hong Kong'}, inplace=True) #reemplazo nombres
    Energy['Energy Supply']=Energy['Energy Supply']*1000000     #paso de peta a giga joules
    Energy=Energy.set_index('Country')
    # print(Energy)
    # print(len(Energy)) #227

    GDP= pd.read_csv('assets/world_bank.csv', header=None, skiprows=4) #elimino header y las 4 priemras filas
    GDP[0]=GDP[0].str.strip()                                   #remuevo espacios de la columna 0
    axislist=[]
    for x in GDP.iloc[0]:                                       #creo lista con los nombres de las cabeceras
        axislist.append(x)
    for k in range (4,len(axislist),1):
        axislist[k]=str(int(axislist[k]))
    GDP.drop([0],axis=0,inplace=True)                           #elimino la fila 0
    GDP.set_axis(axislist, axis=1, inplace=True)                #coloco lista como nuevo axis
    GDP.rename(columns={'Country Name':'Country'}, inplace=True)#cambio nombre de cabecera
    GDP['Country'].replace({"Korea, Rep." : "South Korea", "Iran, Islamic Rep.": "Iran", "Hong Kong SAR, China": "Hong Kong"}, inplace=True)
    GDP=GDP.set_index('Country')
    # print(GDP)
    # print(len(GDP)) #264

    ScimEn= pd.read_excel("assets/scimagojr-3.xlsx", sheet_name = "Sheet1")
    ScimEn=ScimEn.set_index('Country')
    # print(ScimEn)
    # print(len(ScimEn)) #191

    mergedf=pd.merge(ScimEn, Energy, how='outer', left_index=True, right_index=True)  #uno GDP Y Energy dataframes en un solo dataf: mergedf
    df=pd.merge(mergedf, GDP, how='outer', left_index=True, right_index=True)   #uno mergedf con Scim dataframe= todos dataframes en una sola= df
    df=df.drop(df.columns[10:59], axis='columns') #elimino columnas que no necesito
    df.sort_values(by=['Rank'], inplace=True)
    df=df.iloc[0:15]

    lista=[]
    for x in df.columns:                       #quita espacios de las columnas
        str(x).strip()
        lista.append(x)
    df.set_axis(lista, axis=1, inplace=True)  #reestablece los nuevos nombres de columnas
    answer_five=df['Energy Supply per Capita'].mean() #157.6
    print(answer_five)
    print(type(answer_five))
    return(answer_five)


In [14]:
# Cell for autograder.


### Question 6
What country has the maximum % Renewable and what is the percentage?

*This function should return a tuple with the name of the country and the percentage.*

In [5]:
import numpy as np
import re
import pandas as pd
def answer_six():
    Energy= pd.read_excel("assets/Energy Indicators.xls", sheet_name = "Energy")
    Energy=Energy.drop(Energy.columns[[0, 1]], axis='columns')  #elimino columnas por su indice 0 y 1.
    Energy=Energy.drop(range(0,17,1), axis=0)                   #eliino rango de filas 0 a 17 (header)
    Energy=Energy.drop(range(244,282,1), axis=0)                #eliino rango de filas 244 a 282 (footer)
    Energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'] # Renombro columnas
    listaener=[]
    for k in Energy.columns:
        str(k).strip()
        listaener.append(k)
    Energy.set_axis(listaener, axis=1, inplace=True)
    Energy[Energy == '...'] = np.nan                            #relleno con NaN donde haya puntos (...)
    Energy['Country'].replace('[\d*]','', regex=True, inplace=True)  #reemplazar con vacio '' donde cadena de numeros
    Energy['Country'].replace('\(.*\)','', regex=True, inplace=True) #reemplazar con vacio '' donde cadena de numeros ()
    Energy['Country']=Energy['Country'].str.rstrip()            #elimino espacios que sobran al fin de la cadena
    Energy['Country'].replace({'Republic of Korea': 'South Korea', 'United States of America': 'United States',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'China, Hong Kong Special Administrative Region': 'Hong Kong'}, inplace=True) #reemplazo nombres
    Energy['Energy Supply']=Energy['Energy Supply']*1000000     #paso de peta a giga joules
    Energy=Energy.set_index('Country')
    # print(Energy)
    # print(len(Energy)) #227

    GDP= pd.read_csv('assets/world_bank.csv', header=None, skiprows=4) #elimino header y las 4 priemras filas
    GDP[0]=GDP[0].str.strip()                                   #remuevo espacios de la columna 0
    axislist=[]
    for x in GDP.iloc[0]:                                       #creo lista con los nombres de las cabeceras
        axislist.append(x)
    for k in range (4,len(axislist),1):
        axislist[k]=str(int(axislist[k]))
    GDP.drop([0],axis=0,inplace=True)                           #elimino la fila 0
    GDP.set_axis(axislist, axis=1, inplace=True)                #coloco lista como nuevo axis
    GDP.rename(columns={'Country Name':'Country'}, inplace=True)#cambio nombre de cabecera
    GDP['Country'].replace({"Korea, Rep." : "South Korea", "Iran, Islamic Rep.": "Iran", "Hong Kong SAR, China": "Hong Kong"}, inplace=True)
    GDP=GDP.set_index('Country')
    # print(GDP)
    # print(len(GDP)) #264

    ScimEn= pd.read_excel("assets/scimagojr-3.xlsx", sheet_name = "Sheet1")
    ScimEn=ScimEn.set_index('Country')
    # print(ScimEn)
    # print(len(ScimEn)) #191

    mergedf=pd.merge(ScimEn, Energy, how='outer', left_index=True, right_index=True)  #uno GDP Y Energy dataframes en un solo dataf: mergedf
    df=pd.merge(mergedf, GDP, how='outer', left_index=True, right_index=True)   #uno mergedf con Scim dataframe= todos dataframes en una sola= df
    df=df.drop(df.columns[10:59], axis='columns') #elimino columnas que no necesito
    df.sort_values(by=['Rank'], inplace=True)
    df=df.iloc[0:15]

    lista=[]
    for x in df.columns:                       #quita espacios de las columnas
        str(x).strip()
        lista.append(x)
    df.set_axis(lista, axis=1, inplace=True)  #reestablece los nuevos nombres de columnas
    index_max_valor=df.index[df['% Renewable']==df['% Renewable'].max()].tolist() #entrega lista con index que cumpple la condicion
    index_max_valor=index_max_valor[0] #saco el valor de la lista
    valor=df.loc['Brazil']['% Renewable'] #encuentro el valor de Brazil # 69.64803
    answer_six=(index_max_valor, valor)  #armo tupla
    #print(type(answer_six))

    return(answer_six)


In [6]:
assert type(answer_six()) == tuple, "Q6: You should return a tuple!"

assert type(answer_six()[0]) == str, "Q6: The first element in your result should be the name of the country!"


('Brazil', 69.64803)
('Brazil', 69.64803)


### Question 7
Create a new column that is the ratio of Self-Citations to Total Citations. 
What is the maximum value for this new column, and what country has the highest ratio?

*This function should return a tuple with the name of the country and the ratio.*

In [3]:
import numpy as np
import re
import pandas as pd
def answer_seven():
    Energy= pd.read_excel("assets/Energy Indicators.xls", sheet_name = "Energy")
    Energy=Energy.drop(Energy.columns[[0, 1]], axis='columns')  #elimino columnas por su indice 0 y 1.
    Energy=Energy.drop(range(0,17,1), axis=0)                   #eliino rango de filas 0 a 17 (header)
    Energy=Energy.drop(range(244,282,1), axis=0)                #eliino rango de filas 244 a 282 (footer)
    Energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'] # Renombro columnas
    listaener=[]
    for k in Energy.columns:
        str(k).strip()
        listaener.append(k)
    Energy.set_axis(listaener, axis=1, inplace=True)
    Energy[Energy == '...'] = np.nan                            #relleno con NaN donde haya puntos (...)
    Energy['Country'].replace('[\d*]','', regex=True, inplace=True)  #reemplazar con vacio '' donde cadena de numeros
    Energy['Country'].replace('\(.*\)','', regex=True, inplace=True) #reemplazar con vacio '' donde cadena de numeros ()
    Energy['Country']=Energy['Country'].str.rstrip()            #elimino espacios que sobran al fin de la cadena
    Energy['Country'].replace({'Republic of Korea': 'South Korea', 'United States of America': 'United States',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'China, Hong Kong Special Administrative Region': 'Hong Kong'}, inplace=True) #reemplazo nombres
    Energy['Energy Supply']=Energy['Energy Supply']*1000000     #paso de peta a giga joules
    Energy=Energy.set_index('Country')
    # print(Energy)
    # print(len(Energy)) #227

    GDP= pd.read_csv('assets/world_bank.csv', header=None, skiprows=4) #elimino header y las 4 priemras filas
    GDP[0]=GDP[0].str.strip()                                   #remuevo espacios de la columna 0
    axislist=[]
    for x in GDP.iloc[0]:                                       #creo lista con los nombres de las cabeceras
        axislist.append(x)
    for k in range (4,len(axislist),1):
        axislist[k]=str(int(axislist[k]))
    GDP.drop([0],axis=0,inplace=True)                           #elimino la fila 0
    GDP.set_axis(axislist, axis=1, inplace=True)                #coloco lista como nuevo axis
    GDP.rename(columns={'Country Name':'Country'}, inplace=True)#cambio nombre de cabecera
    GDP['Country'].replace({"Korea, Rep." : "South Korea", "Iran, Islamic Rep.": "Iran", "Hong Kong SAR, China": "Hong Kong"}, inplace=True)
    GDP=GDP.set_index('Country')
    # print(GDP)
    # print(len(GDP)) #264

    ScimEn= pd.read_excel("assets/scimagojr-3.xlsx", sheet_name = "Sheet1")
    ScimEn=ScimEn.set_index('Country')
    # print(ScimEn)
    # print(len(ScimEn)) #191

    mergedf=pd.merge(ScimEn, Energy, how='outer', left_index=True, right_index=True)  #uno GDP Y Energy dataframes en un solo dataf: mergedf
    df=pd.merge(mergedf, GDP, how='outer', left_index=True, right_index=True)   #uno mergedf con Scim dataframe= todos dataframes en una sola= df
    df=df.drop(df.columns[10:59], axis='columns') #elimino columnas que no necesito
    df.sort_values(by=['Rank'], inplace=True)
    df=df.iloc[0:15]

    lista=[]
    for x in df.columns:                       #quita espacios de las columnas
        str(x).strip()
        lista.append(x)
    df.set_axis(lista, axis=1, inplace=True)  #reestablece los nuevos nombres de columnas


    df['Ratio_self-citations']=df['Self-citations']/df['Citations']
    index_max_valor=df.index[df['Ratio_self-citations']==df['Ratio_self-citations'].max()].tolist() #entrega lista con index que cumpple la condicion
    index_max_valor=index_max_valor[0] #saco el valor de la lista
    
    valor=df.loc['China']['Ratio_self-citations'] #encuentro el valor de china
    answer_seven=(index_max_valor, valor)         #armo tupla

    print(answer_seven)
    return(answer_seven)


In [4]:
assert type(answer_seven()) == tuple, "Q7: You should return a tuple!"

assert type(answer_seven()[0]) == str, "Q7: The first element in your result should be the name of the country!"


('China', 0.6893126179389422)
('China', 0.6893126179389422)


### Question 8

Create a column that estimates the population using Energy Supply and Energy Supply per capita. 
What is the third most populous country according to this estimate?

*This function should return the name of the country*

In [13]:
import numpy as np
import re
import pandas as pd
def answer_eight():
    Energy= pd.read_excel("assets/Energy Indicators.xls", sheet_name = "Energy")
    Energy=Energy.drop(Energy.columns[[0, 1]], axis='columns')  #elimino columnas por su indice 0 y 1.
    Energy=Energy.drop(range(0,17,1), axis=0)                   #eliino rango de filas 0 a 17 (header)
    Energy=Energy.drop(range(244,282,1), axis=0)                #eliino rango de filas 244 a 282 (footer)
    Energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'] # Renombro columnas
    listaener=[]
    for k in Energy.columns:
        str(k).strip()
        listaener.append(k)
    Energy.set_axis(listaener, axis=1, inplace=True)
    Energy[Energy == '...'] = np.nan                            #relleno con NaN donde haya puntos (...)
    Energy['Country'].replace('[\d*]','', regex=True, inplace=True)  #reemplazar con vacio '' donde cadena de numeros
    Energy['Country'].replace('\(.*\)','', regex=True, inplace=True) #reemplazar con vacio '' donde cadena de numeros ()
    Energy['Country']=Energy['Country'].str.rstrip()            #elimino espacios que sobran al fin de la cadena
    Energy['Country'].replace({'Republic of Korea': 'South Korea', 'United States of America': 'United States',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'China, Hong Kong Special Administrative Region': 'Hong Kong'}, inplace=True) #reemplazo nombres
    Energy['Energy Supply']=Energy['Energy Supply']*1000000     #paso de peta a giga joules
    Energy=Energy.set_index('Country')
    # print(Energy)
    # print(len(Energy)) #227

    GDP= pd.read_csv('assets/world_bank.csv', header=None, skiprows=4) #elimino header y las 4 priemras filas
    GDP[0]=GDP[0].str.strip()                                   #remuevo espacios de la columna 0
    axislist=[]
    for x in GDP.iloc[0]:                                       #creo lista con los nombres de las cabeceras
        axislist.append(x)
    for k in range (4,len(axislist),1):
        axislist[k]=str(int(axislist[k]))
    GDP.drop([0],axis=0,inplace=True)                           #elimino la fila 0
    GDP.set_axis(axislist, axis=1, inplace=True)                #coloco lista como nuevo axis
    GDP.rename(columns={'Country Name':'Country'}, inplace=True)#cambio nombre de cabecera
    GDP['Country'].replace({"Korea, Rep." : "South Korea", "Iran, Islamic Rep.": "Iran", "Hong Kong SAR, China": "Hong Kong"}, inplace=True)
    GDP=GDP.set_index('Country')
    # print(GDP)
    # print(len(GDP)) #264

    ScimEn= pd.read_excel("assets/scimagojr-3.xlsx", sheet_name = "Sheet1")
    ScimEn=ScimEn.set_index('Country')
    # print(ScimEn)
    # print(len(ScimEn)) #191

    mergedf=pd.merge(ScimEn, Energy, how='outer', left_index=True, right_index=True)  #uno GDP Y Energy dataframes en un solo dataf: mergedf
    df=pd.merge(mergedf, GDP, how='outer', left_index=True, right_index=True)   #uno mergedf con Scim dataframe= todos dataframes en una sola= df
    df=df.drop(df.columns[10:59], axis='columns') #elimino columnas que no necesito
    df.sort_values(by=['Rank'], inplace=True)
    df=df.iloc[0:15]

    lista=[]
    for x in df.columns:                       #quita espacios de las columnas
        str(x).strip()
        lista.append(x)
    df.set_axis(lista, axis=1, inplace=True)  #reestablece los nuevos nombres de columnas

    df['Population']=df['Energy Supply']/df['Energy Supply per Capita']
    df.sort_values(by=['Population'], inplace=True, ascending=False)

    index=df.index           #lista de index en orden descendente
    answer_eight=index[2]   #tercer index
    #print(answer_eight)
      
    return(answer_eight)
answer_eight()

'United States'

In [14]:
assert type(answer_eight()) == str, "Q8: You should return the name of the country!"


### Question 9
Create a column that estimates the number of citable documents per person. 
What is the correlation between the number of citable documents per capita and the energy supply per capita? Use the `.corr()` method, (Pearson's correlation).

*This function should return a single number.*

*(Optional: Use the built-in function `plot9()` to visualize the relationship between Energy Supply per Capita vs. Citable docs per Capita)*

In [7]:
import numpy as np
import re
import pandas as pd
import scipy.stats as stats
def answer_nine():
    Energy= pd.read_excel("assets/Energy Indicators.xls", sheet_name = "Energy")
    Energy=Energy.drop(Energy.columns[[0, 1]], axis='columns')  #elimino columnas por su indice 0 y 1.
    Energy=Energy.drop(range(0,17,1), axis=0)                   #eliino rango de filas 0 a 17 (header)
    Energy=Energy.drop(range(244,282,1), axis=0)                #eliino rango de filas 244 a 282 (footer)
    Energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'] # Renombro columnas
    listaener=[]
    for k in Energy.columns:
        str(k).strip()
        listaener.append(k)
    Energy.set_axis(listaener, axis=1, inplace=True)
    Energy[Energy == '...'] = np.nan                            #relleno con NaN donde haya puntos (...)
    Energy['Country'].replace('[\d*]','', regex=True, inplace=True)  #reemplazar con vacio '' donde cadena de numeros
    Energy['Country'].replace('\(.*\)','', regex=True, inplace=True) #reemplazar con vacio '' donde cadena de numeros ()
    Energy['Country']=Energy['Country'].str.rstrip()            #elimino espacios que sobran al fin de la cadena
    Energy['Country'].replace({'Republic of Korea': 'South Korea', 'United States of America': 'United States',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'China, Hong Kong Special Administrative Region': 'Hong Kong'}, inplace=True) #reemplazo nombres
    Energy['Energy Supply']=Energy['Energy Supply']*1000000     #paso de peta a giga joules
    Energy=Energy.set_index('Country')
    # print(Energy)
    # print(len(Energy)) #227

    GDP= pd.read_csv('assets/world_bank.csv', header=None, skiprows=4) #elimino header y las 4 priemras filas
    GDP[0]=GDP[0].str.strip()                                   #remuevo espacios de la columna 0
    axislist=[]
    for x in GDP.iloc[0]:                                       #creo lista con los nombres de las cabeceras
        axislist.append(x)
    for k in range (4,len(axislist),1):
        axislist[k]=str(int(axislist[k]))
    GDP.drop([0],axis=0,inplace=True)                           #elimino la fila 0
    GDP.set_axis(axislist, axis=1, inplace=True)                #coloco lista como nuevo axis
    GDP.rename(columns={'Country Name':'Country'}, inplace=True)#cambio nombre de cabecera
    GDP['Country'].replace({"Korea, Rep." : "South Korea", "Iran, Islamic Rep.": "Iran", "Hong Kong SAR, China": "Hong Kong"}, inplace=True)
    GDP=GDP.set_index('Country')
    # print(GDP)
    # print(len(GDP)) #264

    ScimEn= pd.read_excel("assets/scimagojr-3.xlsx", sheet_name = "Sheet1")
    ScimEn=ScimEn.set_index('Country')
    # print(ScimEn)
    # print(len(ScimEn)) #191

    mergedf=pd.merge(ScimEn, Energy, how='outer', left_index=True, right_index=True)  #uno GDP Y Energy dataframes en un solo dataf: mergedf
    df=pd.merge(mergedf, GDP, how='outer', left_index=True, right_index=True)   #uno mergedf con Scim dataframe= todos dataframes en una sola= df
    df=df.drop(df.columns[10:59], axis='columns') #elimino columnas que no necesito
    df.sort_values(by=['Rank'], inplace=True)
    df=df.iloc[0:15]

    lista=[]
    for x in df.columns:                       #quita espacios de las columnas
        str(x).strip()
        lista.append(x)
    df.set_axis(lista, axis=1, inplace=True)  #reestablece los nuevos nombres de columnas

    df['Population']=df['Energy Supply']/df['Energy Supply per Capita']
    df['Citable docs per Capita'] = df['Citable documents'] / df['Population']
    corr, pval=stats.pearsonr(df['Citable docs per Capita'],df["Energy Supply per Capita"])  #se devuelven dos variables coor, pval
    #print('Correlacion: ',corr)
    #print('pval:        ', pval)

    
    answer_nine=corr 
    return(answer_nine)
answer_nine()

0.7940010435442943

In [8]:
def plot9():
    import matplotlib as plt
    %matplotlib inline
    
    Top15 = answer_one()
    Top15['PopEst'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']
    Top15['Citable docs per Capita'] = Top15['Citable documents'] / Top15['PopEst']
    Top15.plot(x='Citable docs per Capita', y='Energy Supply per Capita', kind='scatter', xlim=[0, 0.0006])

In [3]:
assert answer_nine() >= -1. and answer_nine() <= 1., "Q9: A valid correlation should between -1 to 1!"


Correlacion:  0.7940010435442943
pval:         0.0004083648953039715
Correlacion:  0.7940010435442943
pval:         0.0004083648953039715


### Question 10
Create a new column with a 1 if the country's % Renewable value is at or above the median for all countries in the top 15, and a 0 if the country's % Renewable value is below the median.

*This function should return a series named `HighRenew` whose index is the country name sorted in ascending order of rank.*

In [11]:
import numpy as np
import re
import pandas as pd
import scipy.stats as stats
def answer_ten():
    Energy= pd.read_excel("assets/Energy Indicators.xls", sheet_name = "Energy")
    Energy=Energy.drop(Energy.columns[[0, 1]], axis='columns')  #elimino columnas por su indice 0 y 1.
    Energy=Energy.drop(range(0,17,1), axis=0)                   #eliino rango de filas 0 a 17 (header)
    Energy=Energy.drop(range(244,282,1), axis=0)                #eliino rango de filas 244 a 282 (footer)
    Energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'] # Renombro columnas
    listaener=[]
    for k in Energy.columns:
        str(k).strip()
        listaener.append(k)
    Energy.set_axis(listaener, axis=1, inplace=True)
    Energy[Energy == '...'] = np.nan                            #relleno con NaN donde haya puntos (...)
    Energy['Country'].replace('[\d*]','', regex=True, inplace=True)  #reemplazar con vacio '' donde cadena de numeros
    Energy['Country'].replace('\(.*\)','', regex=True, inplace=True) #reemplazar con vacio '' donde cadena de numeros ()
    Energy['Country']=Energy['Country'].str.rstrip()            #elimino espacios que sobran al fin de la cadena
    Energy['Country'].replace({'Republic of Korea': 'South Korea', 'United States of America': 'United States',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'China, Hong Kong Special Administrative Region': 'Hong Kong'}, inplace=True) #reemplazo nombres
    Energy['Energy Supply']=Energy['Energy Supply']*1000000     #paso de peta a giga joules
    Energy=Energy.set_index('Country')
    # print(Energy)
    # print(len(Energy)) #227

    GDP= pd.read_csv('assets/world_bank.csv', header=None, skiprows=4) #elimino header y las 4 priemras filas
    GDP[0]=GDP[0].str.strip()                                   #remuevo espacios de la columna 0
    axislist=[]
    for x in GDP.iloc[0]:                                       #creo lista con los nombres de las cabeceras
        axislist.append(x)
    for k in range (4,len(axislist),1):
        axislist[k]=str(int(axislist[k]))
    GDP.drop([0],axis=0,inplace=True)                           #elimino la fila 0
    GDP.set_axis(axislist, axis=1, inplace=True)                #coloco lista como nuevo axis
    GDP.rename(columns={'Country Name':'Country'}, inplace=True)#cambio nombre de cabecera
    GDP['Country'].replace({"Korea, Rep." : "South Korea", "Iran, Islamic Rep.": "Iran", "Hong Kong SAR, China": "Hong Kong"}, inplace=True)
    GDP=GDP.set_index('Country')
    # print(GDP)
    # print(len(GDP)) #264

    ScimEn= pd.read_excel("assets/scimagojr-3.xlsx", sheet_name = "Sheet1")
    ScimEn=ScimEn.set_index('Country')
    # print(ScimEn)
    # print(len(ScimEn)) #191

    mergedf=pd.merge(ScimEn, Energy, how='outer', left_index=True, right_index=True)  #uno GDP Y Energy dataframes en un solo dataf: mergedf
    df=pd.merge(mergedf, GDP, how='outer', left_index=True, right_index=True)   #uno mergedf con Scim dataframe= todos dataframes en una sola= df
    df=df.drop(df.columns[10:59], axis='columns') #elimino columnas que no necesito
    df.sort_values(by=['Rank'], inplace=True)
    df=df.iloc[0:15]

    lista=[]
    for x in df.columns:                       #quita espacios de las columnas
        str(x).strip()
        lista.append(x)
    df.set_axis(lista, axis=1, inplace=True)  #reestablece los nuevos nombres de columnas

    df['HighRenew']=np.where(df['% Renewable'] >= df['% Renewable'].median(), 1 , 0)

    answer_ten=df['HighRenew']
    print(type(answer_ten))
    
    return(answer_ten)




In [12]:
assert type(answer_ten()) == pd.Series, "Q10: You should return a Series!"


<class 'pandas.core.series.Series'>


### Question 11
Use the following dictionary to group the Countries by Continent, then create a DataFrame that displays the sample size (the number of countries in each continent bin), and the sum, mean, and std deviation for the estimated population of each country.

```python
ContinentDict  = {'China':'Asia', 
                  'United States':'North America', 
                  'Japan':'Asia', 
                  'United Kingdom':'Europe', 
                  'Russian Federation':'Europe', 
                  'Canada':'North America', 
                  'Germany':'Europe', 
                  'India':'Asia',
                  'France':'Europe', 
                  'South Korea':'Asia', 
                  'Italy':'Europe', 
                  'Spain':'Europe', 
                  'Iran':'Asia',
                  'Australia':'Australia', 
                  'Brazil':'South America'}
```

*This function should return a DataFrame with index named Continent `['Asia', 'Australia', 'Europe', 'North America', 'South America']` and columns `['size', 'sum', 'mean', 'std']`*

In [9]:
import numpy as np
import re
import pandas as pd
def answer_eleven():
        Energy= pd.read_excel("assets/Energy Indicators.xls", sheet_name = "Energy")
        Energy=Energy.drop(Energy.columns[[0, 1]], axis='columns')  #elimino columnas por su indice 0 y 1.
        Energy=Energy.drop(range(0,17,1), axis=0)                   #eliino rango de filas 0 a 17 (header)
        Energy=Energy.drop(range(244,282,1), axis=0)                #eliino rango de filas 244 a 282 (footer)
        Energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'] # Renombro columnas
        listaener=[]
        for k in Energy.columns:
            str(k).strip()
            listaener.append(k)
        Energy.set_axis(listaener, axis=1, inplace=True)
        Energy[Energy == '...'] = np.nan                            #relleno con NaN donde haya puntos (...)
        Energy['Country'].replace('[\d*]','', regex=True, inplace=True)  #reemplazar con vacio '' donde cadena de numeros
        Energy['Country'].replace('\(.*\)','', regex=True, inplace=True) #reemplazar con vacio '' donde cadena de numeros ()
        Energy['Country']=Energy['Country'].str.rstrip()            #elimino espacios que sobran al fin de la cadena
        Energy['Country'].replace({'Republic of Korea': 'South Korea', 'United States of America': 'United States',
        'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
        'China, Hong Kong Special Administrative Region': 'Hong Kong'}, inplace=True) #reemplazo nombres
        Energy['Energy Supply']=Energy['Energy Supply']*1000000     #paso de peta a giga joules
        Energy=Energy.set_index('Country')
        # print(Energy)
        # print(len(Energy)) #227

        GDP= pd.read_csv('assets/world_bank.csv', header=None, skiprows=4) #elimino header y las 4 priemras filas
        GDP[0]=GDP[0].str.strip()                                   #remuevo espacios de la columna 0
        axislist=[]
        for x in GDP.iloc[0]:                                       #creo lista con los nombres de las cabeceras
            axislist.append(x)
        for k in range (4,len(axislist),1):
            axislist[k]=str(int(axislist[k]))
        GDP.drop([0],axis=0,inplace=True)                           #elimino la fila 0
        GDP.set_axis(axislist, axis=1, inplace=True)                #coloco lista como nuevo axis
        GDP.rename(columns={'Country Name':'Country'}, inplace=True)#cambio nombre de cabecera
        GDP['Country'].replace({"Korea, Rep." : "South Korea", "Iran, Islamic Rep.": "Iran", "Hong Kong SAR, China": "Hong Kong"}, inplace=True)
        GDP=GDP.set_index('Country')
        # print(GDP)
        # print(len(GDP)) #264

        ScimEn= pd.read_excel("assets/scimagojr-3.xlsx", sheet_name = "Sheet1")
        ScimEn=ScimEn.set_index('Country')
        # print(ScimEn)
        # print(len(ScimEn)) #191

        mergedf=pd.merge(ScimEn, Energy, how='outer', left_index=True, right_index=True)  #uno GDP Y Energy dataframes en un solo dataf: mergedf
        df=pd.merge(mergedf, GDP, how='outer', left_index=True, right_index=True)   #uno mergedf con Scim dataframe= todos dataframes en una sola= df
        df=df.drop(df.columns[10:59], axis='columns') #elimino columnas que no necesito
        df.sort_values(by=['Rank'], inplace=True)
        df=df.iloc[0:15]

        lista=[]
        for x in df.columns:                       #quita espacios de las columnas
            str(x).strip()
            lista.append(x)
        df.set_axis(lista, axis=1, inplace=True)  #reestablece los nuevos nombres de columnas

        df['Population']=df['Energy Supply']/df['Energy Supply per Capita']
        ContinentDict  = {'China':'Asia',
                      'United States':'North America',
                      'Japan':'Asia',
                      'United Kingdom':'Europe',
                      'Russian Federation':'Europe',
                      'Canada':'North America',
                      'Germany':'Europe',
                      'India':'Asia',
                      'France':'Europe',
                      'South Korea':'Asia',
                      'Italy':'Europe',
                      'Spain':'Europe',
                      'Iran':'Asia',
                      'Australia':'Australia',
                      'Brazil':'South America'}
        continents=pd.Series(ContinentDict)
        df['Continent']=continents
        list_continents=continents.unique()
        list_continents.sort()                        #lista continentes

        df.reset_index(inplace = True)
        df.set_index(['Continent','Country'], inplace=True)
        df.sort_values(by=['Continent'], inplace=True)

        size=[]
        sum=[]
        mean=[]
        std=[]
        for i in list_continents:
            size.append(len(df.loc[i]))                 #lista de size
            sum.append(df.loc[i]['Population'].sum())   #lista de sum  population
            mean.append(df.loc[i]['Population'].mean()) #lista de mean population
            std.append(df.loc[i]['Population'].std())   #lista de std  population

        col = {"size": size, "sum": sum, "mean": mean, "std":std}
        answer_eleven=pd.DataFrame(col, index=list_continents)


        return(answer_eleven)


In [10]:
assert type(answer_eleven()) == pd.DataFrame, "Q11: You should return a DataFrame!"

assert answer_eleven().shape[0] == 5, "Q11: Wrong row numbers!"

assert answer_eleven().shape[1] == 4, "Q11: Wrong column numbers!"


### Question 12
Cut % Renewable into 5 bins. Group Top15 by the Continent, as well as these new % Renewable bins. How many countries are in each of these groups?

*This function should return a Series with a MultiIndex of `Continent`, then the bins for `% Renewable`. Do not include groups with no countries.*

In [ ]:
import numpy as np
import re
import pandas as pd
def answer_twelve():
        Energy= pd.read_excel("assets/Energy Indicators.xls", sheet_name = "Energy")
        Energy=Energy.drop(Energy.columns[[0, 1]], axis='columns')  #elimino columnas por su indice 0 y 1.
        Energy=Energy.drop(range(0,17,1), axis=0)                   #eliino rango de filas 0 a 17 (header)
        Energy=Energy.drop(range(244,282,1), axis=0)                #eliino rango de filas 244 a 282 (footer)
        Energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'] # Renombro columnas
        listaener=[]
        for k in Energy.columns:
            str(k).strip()
            listaener.append(k)
        Energy.set_axis(listaener, axis=1, inplace=True)
        Energy[Energy == '...'] = np.nan                            #relleno con NaN donde haya puntos (...)
        Energy['Country'].replace('[\d*]','', regex=True, inplace=True)  #reemplazar con vacio '' donde cadena de numeros
        Energy['Country'].replace('\(.*\)','', regex=True, inplace=True) #reemplazar con vacio '' donde cadena de numeros ()
        Energy['Country']=Energy['Country'].str.rstrip()            #elimino espacios que sobran al fin de la cadena
        Energy['Country'].replace({'Republic of Korea': 'South Korea', 'United States of America': 'United States',
        'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
        'China, Hong Kong Special Administrative Region': 'Hong Kong'}, inplace=True) #reemplazo nombres
        Energy['Energy Supply']=Energy['Energy Supply']*1000000     #paso de peta a giga joules
        Energy=Energy.set_index('Country')
        # print(Energy)
        # print(len(Energy)) #227

        GDP= pd.read_csv('assets/world_bank.csv', header=None, skiprows=4) #elimino header y las 4 priemras filas
        GDP[0]=GDP[0].str.strip()                                   #remuevo espacios de la columna 0
        axislist=[]
        for x in GDP.iloc[0]:                                       #creo lista con los nombres de las cabeceras
            axislist.append(x)
        for k in range (4,len(axislist),1):
            axislist[k]=str(int(axislist[k]))
        GDP.drop([0],axis=0,inplace=True)                           #elimino la fila 0
        GDP.set_axis(axislist, axis=1, inplace=True)                #coloco lista como nuevo axis
        GDP.rename(columns={'Country Name':'Country'}, inplace=True)#cambio nombre de cabecera
        GDP['Country'].replace({"Korea, Rep." : "South Korea", "Iran, Islamic Rep.": "Iran", "Hong Kong SAR, China": "Hong Kong"}, inplace=True)
        GDP=GDP.set_index('Country')
        # print(GDP)
        # print(len(GDP)) #264

        ScimEn= pd.read_excel("assets/scimagojr-3.xlsx", sheet_name = "Sheet1")
        ScimEn=ScimEn.set_index('Country')
        # print(ScimEn)
        # print(len(ScimEn)) #191

        mergedf=pd.merge(ScimEn, Energy, how='outer', left_index=True, right_index=True)  #uno GDP Y Energy dataframes en un solo dataf: mergedf
        df=pd.merge(mergedf, GDP, how='outer', left_index=True, right_index=True)   #uno mergedf con Scim dataframe= todos dataframes en una sola= df
        df=df.drop(df.columns[10:59], axis='columns') #elimino columnas que no necesito
        df.sort_values(by=['Rank'], inplace=True)
        df=df.iloc[0:15]

        lista=[]
        for x in df.columns:                       #quita espacios de las columnas
            str(x).strip()
            lista.append(x)
        df.set_axis(lista, axis=1, inplace=True)  #reestablece los nuevos nombres de columnas

        ContinentDict  = {'China':'Asia',
                      'United States':'North America',
                      'Japan':'Asia',
                      'United Kingdom':'Europe',
                      'Russian Federation':'Europe',
                      'Canada':'North America',
                      'Germany':'Europe',
                      'India':'Asia',
                      'France':'Europe',
                      'South Korea':'Asia',
                      'Italy':'Europe',
                      'Spain':'Europe',
                      'Iran':'Asia',
                      'Australia':'Australia',
                      'Brazil':'South America'}
        continents=pd.Series(ContinentDict)
        df['Continent']=continents
        list_continents=continents.unique()
        list_continents.sort()               #lista continentes


        df.reset_index(inplace = True)
        df.set_index(['Continent','Country'], inplace=True)
        df.sort_values(by=['Continent'], inplace=True)

        print(df)
        return(answer_twelve)
answer_twelve()


In [ ]:
assert type(answer_twelve()) == pd.Series, "Q12: You should return a Series!"

assert len(answer_twelve()) == 9, "Q12: Wrong result numbers!"


### Question 13
Convert the Population Estimate series to a string with thousands separator (using commas). Use all significant digits (do not round the results).

e.g. 12345678.90 -> 12,345,678.90

*This function should return a series `PopEst` whose index is the country name and whose values are the population estimate string*

In [ ]:
def answer_thirteen():
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
assert type(answer_thirteen()) == pd.Series, "Q13: You should return a Series!"

assert len(answer_thirteen()) == 15, "Q13: Wrong result numbers!"


### Optional

Use the built in function `plot_optional()` to see an example visualization.

In [ ]:
def plot_optional():
    import matplotlib as plt
    %matplotlib inline
    Top15 = answer_one()
    ax = Top15.plot(x='Rank', y='% Renewable', kind='scatter', 
                    c=['#e41a1c','#377eb8','#e41a1c','#4daf4a','#4daf4a','#377eb8','#4daf4a','#e41a1c',
                       '#4daf4a','#e41a1c','#4daf4a','#4daf4a','#e41a1c','#dede00','#ff7f00'], 
                    xticks=range(1,16), s=6*Top15['2014']/10**10, alpha=.75, figsize=[16,6]);

    for i, txt in enumerate(Top15.index):
        ax.annotate(txt, [Top15['Rank'][i], Top15['% Renewable'][i]], ha='center')

    print("This is an example of a visualization that can be created to help understand the data. \
This is a bubble chart showing % Renewable vs. Rank. The size of the bubble corresponds to the countries' \
2014 GDP, and the color corresponds to the continent.")

## 📌 Week 4 — Sports Data & Hypothesis Testing

> **Datasets:** `nhl.csv`, `nba.csv`, `mlb.csv`, `nfl.csv`, `wikipedia_data.html`  
> **Goal:** Correlate win/loss ratios with city population across 4 major sports leagues and test statistical hypotheses.

---

# Assignment 4
## Description
In this assignment you must read in a file of metropolitan regions and associated sports teams from [assets/wikipedia_data.html](assets/wikipedia_data.html) and answer some questions about each metropolitan region. Each of these regions may have one or more teams from the "Big 4": NFL (football, in [assets/nfl.csv](assets/nfl.csv)), MLB (baseball, in [assets/mlb.csv](assets/mlb.csv)), NBA (basketball, in [assets/nba.csv](assets/nba.csv) or NHL (hockey, in [assets/nhl.csv](assets/nhl.csv)). Please keep in mind that all questions are from the perspective of the metropolitan region, and that this file is the "source of authority" for the location of a given sports team. Thus teams which are commonly known by a different area (e.g. "Oakland Raiders") need to be mapped into the metropolitan region given (e.g. San Francisco Bay Area). This will require some human data understanding outside of the data you've been given (e.g. you will have to hand-code some names, and might need to google to find out where teams are)!

For each sport I would like you to answer the question: **what is the win/loss ratio's correlation with the population of the city it is in?** Win/Loss ratio refers to the number of wins over the number of wins plus the number of losses. Remember that to calculate the correlation with [`pearsonr`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html), so you are going to send in two ordered lists of values, the populations from the wikipedia_data.html file and the win/loss ratio for a given sport in the same order. Average the win/loss ratios for those cities which have multiple teams of a single sport. Each sport is worth an equal amount in this assignment (20%\*4=80%) of the grade for this assignment. You should only use data **from year 2018** for your analysis -- this is important!

## Notes

1. Do not include data about the MLS or CFL in any of the work you are doing, we're only interested in the Big 4 in this assignment.
2. I highly suggest that you first tackle the four correlation questions in order, as they are all similar and worth the majority of grades for this assignment. This is by design!
3. It's fair game to talk with peers about high level strategy as well as the relationship between metropolitan areas and sports teams. However, do not post code solving aspects of the assignment (including such as dictionaries mapping areas to teams, or regexes which will clean up names).
4. There may be more teams than the assert statements test, remember to collapse multiple teams in one city into a single value!

## Question 1
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NHL** using **2018** data.

In [5]:
def nhl_correlation(): 
    
    import pandas as pd
    import numpy as np
    import scipy.stats as stats
    import re
    import pprint

    cities=pd.read_html("assets/wikipedia_data.html")[1]
    cities=cities.iloc[:-1,[0,3,5,6,7,8]]                   #cargo las columnas que necesito
    cities.replace('\[\w.*\]','', regex=True, inplace=True) # reemplazo [] por espacio ''
    cities.replace('—',0, regex=True, inplace=True)         # reemplazo — por 0 = NaN
    cities.replace('',0, regex=True, inplace=True)          # reemplazo '' por 0 = NaN
    cities['NHL']=cities['NHL'].str.strip()                 # retiro espacios
    cities['NFL']=cities['NFL'].str.strip()
    cities['MLB']=cities['MLB'].str.strip()
    cities['NBA']=cities['NBA'].str.strip()
    copy_NHL=cities[['Metropolitan area','NHL','Population (2016 est.)[8]']].dropna()
    copy_NHL.sort_values(by=['NHL'], inplace=True)
    copy_NHL['Population (2016 est.)[8]']=copy_NHL['Population (2016 est.)[8]'].astype('int64')

    population_by_region=copy_NHL['Population (2016 est.)[8]']
  


    nhl_df=pd.read_csv("assets/nhl.csv")
    nhl_df=nhl_df.iloc[:35,[0,2,3,13,14]] #cargo las columnas que necesito
    nhl_df=nhl_df.drop(nhl_df.index[[0, 9, 18, 26]], axis=0) #elimino filas que no necesito
    nhl_df['team'].replace("\*",'',inplace=True,regex=True)  # reemplazo * por espacio"
    nhl_df['team'].replace("[\D].*\s",'',inplace=True,regex=True)
    nhl_df['team'].replace({'Knights' : 'Golden Knights', 'Jackets': 'Blue Jackets', 'Leafs': 'Maple Leafs', 'Wings': 'Red Wings' }, inplace=True)
    nhl_df['team']=nhl_df['team'].str.strip()                #retiro espacios
    nhl_df['W']=nhl_df['W'].astype('int64')                  #convierto columna str a int para poder dividir
    nhl_df['L']=nhl_df['L'].astype('int64')                  #convierto columna str a int para poder dividir
    nhl_df['W/L Ratio']=nhl_df['W']/(nhl_df['W']+nhl_df['L'])  #convierto columna str a int
    nhl_df=nhl_df.drop(nhl_df.columns[[1, 2, 3, 4]], axis=1) #elimino columnas que no necesito
    nhl_df['team'].replace({'Rangers' : 'RangersIslandersDevils'}, inplace=True)
    nhl_df.iloc[15,1]=(nhl_df.iloc[15,1]+nhl_df.iloc[14,1]+nhl_df.iloc[12,1])/3     #RangersIslandersDevils promedio de los 3
    nhl_df=nhl_df.drop(nhl_df.index[[14, 12]], axis=0)                              #elimino filas Islanders y Devils
    nhl_df['team'].replace({'Kings' : 'KingsDucks'}, inplace=True)
    nhl_df.iloc[24,1]=(nhl_df.iloc[24,1]+nhl_df.iloc[22,1])/2     #KingsDucks promedio de los 2
    nhl_df=nhl_df.drop(nhl_df.index[[22]], axis=0)
    nhl_df.sort_values(by=['team'], inplace=True)

    win_loss_by_region = nhl_df['W/L Ratio']
    
    corr, pval= stats.pearsonr(population_by_region, win_loss_by_region)
    
    

    assert len(population_by_region) == len(win_loss_by_region), "Q1: Your lists must be the same length"
    assert len(population_by_region) == 28, "Q1: There should be 28 teams being analysed for NHL"
    
    
    return corr
nhl_correlation()

0.012486162921209909

## Question 2
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NBA** using **2018** data.

In [8]:
def nba_correlation():
    import pandas as pd
    import numpy as np
    import scipy.stats as stats
    import re

    cities=pd.read_html("assets/wikipedia_data.html")[1]
    cities=cities.iloc[:-1,[0,3,5,6,7,8]]
    cities.replace('\[\w.*\]','', regex=True, inplace=True) # reemplazo [] por espacio ''
    cities.replace('—',0, regex=True, inplace=True)         # reemplazo — por 0 = NaN
    cities.replace('',0, regex=True, inplace=True)          # reemplazo '' por 0 = NaN
    cities['NBA']=cities['NBA'].str.strip()
    copy_NBA=cities[['Metropolitan area','NBA','Population (2016 est.)[8]']].dropna()
    copy_NBA.sort_values(by=['NBA'], inplace=True)

    copy_NBA['Population (2016 est.)[8]']=copy_NBA['Population (2016 est.)[8]'].astype('int64')


    population_by_region=copy_NBA['Population (2016 est.)[8]']

    nba_df=pd.read_csv("assets/nba.csv")

    nba_df=nba_df.iloc[:30,[0,3]] #cargo las columnas que necesito
    # nba_df['W/L%']=nba_df['W/L%'].astype('int64')
    nba_df.rename(columns={"W/L%": "W/L Ratio"}, inplace=True)
    nba_df['W/L Ratio']=nba_df['W/L Ratio'].astype('float64')
    nba_df.replace('\(\d.*\)','', regex=True, inplace=True) # reemplazo () por espacio ''
    nba_df.replace('\*','', regex=True, inplace=True) # reemplazo () por espacio ''
    nba_df['team']=nba_df['team'].str.strip()
    nba_df['team'].replace("[\D].*\s",'',inplace=True,regex=True)


    nba_df['team'].replace({'Blazers' : 'Trail Blazers', 'Clippers': 'LakersClippers', 'Knicks': 'KnicksNets'}, inplace=True)
    nba_df.iloc[24,1]=(nba_df.iloc[24,1]+nba_df.iloc[25,1])/2     #KingsDucks promedio de los 2
    nba_df.iloc[10,1]=(nba_df.iloc[10,1]+nba_df.iloc[11,1])/2     #KingsDucks promedio de los 2
    nba_df=nba_df.drop(nba_df.index[[11, 25]], axis=0)
    nba_df.sort_values(by=['team'], inplace=True)

    win_loss_by_region = nba_df['W/L Ratio']
    corr, pval= stats.pearsonr(population_by_region, win_loss_by_region)
    print(corr)
    return corr
nba_correlation()

-0.17636350642182938


-0.17636350642182938

## Question 3
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **MLB** using **2018** data.

In [1]:
def mlb_correlation():
    import pandas as pd
    import numpy as np
    import scipy.stats as stats
    import re

    cities=pd.read_html("assets/wikipedia_data.html")[1]
    cities=cities.iloc[:-1,[0,3,5,6,7,8]]
    cities.replace('\[\w.*\]','', regex=True, inplace=True) # reemplazo [] por espacio ''
    cities.replace('—',0, regex=True, inplace=True)         # reemplazo — por 0 = NaN
    cities.replace('',0, regex=True, inplace=True)          # reemplazo '' por 0 = NaN
    cities['MLB']=cities['MLB'].str.strip()
    copy_MLB=cities[['Metropolitan area','MLB','Population (2016 est.)[8]']].dropna()
    copy_MLB.sort_values(by=['MLB'], inplace=True)  #len=26
    copy_MLB['Population (2016 est.)[8]']=copy_MLB['Population (2016 est.)[8]'].astype('int64')

    population_by_region=copy_MLB['Population (2016 est.)[8]']


    mlb_df=pd.read_csv("assets/mlb.csv")
    mlb_df=mlb_df.iloc[:30,[0,1,2]] #cargo las columnas que necesito
    
    mlb_df['W']=mlb_df['W'].astype('int64')                  #convierto columna str a int para poder dividir
    mlb_df['L']=mlb_df['L'].astype('int64')                  #convierto columna str a int para poder dividir
    mlb_df['W/L Ratio']=mlb_df['W']/(mlb_df['W']+mlb_df['L'])  #convierto columna str a int


    mlb_df['team'].replace("[\D].*\s",'',inplace=True,regex=True)
    mlb_df['team']=mlb_df['team'].str.strip()
    mlb_df.iloc[0,0]='Red Sox'
    mlb_df.iloc[8,0]='White Sox'
    mlb_df['team'].replace({'Yankees' : 'YankeesMets', 'Dodgers': 'DodgersAngels', 'Giants': 'GiantsAthletics', 'Cubs': 'CubsWhite Sox', 'Jays': 'Blue Jays'}, inplace=True)
    mlb_df.iloc[1,3]=(mlb_df.iloc[1,3] + mlb_df.iloc[18,3])/2
    mlb_df.iloc[25,3]=(mlb_df.iloc[25,3] + mlb_df.iloc[13,3])/2
    mlb_df.iloc[28,3]=(mlb_df.iloc[28,3] + mlb_df.iloc[11,3])/2
    mlb_df.iloc[21,3]=(mlb_df.iloc[21,3] + mlb_df.iloc[8,3])/2

    mlb_df=mlb_df.drop(mlb_df.index[[18,13,11,8]], axis=0)
    mlb_df.sort_values(by=['team'], inplace=True)

    win_loss_by_region = mlb_df['W/L Ratio']

    corr, pval= stats.pearsonr(population_by_region, win_loss_by_region)
    print(corr)
    return corr
    
mlb_correlation()

0.1502769830266931


0.1502769830266931

## Question 4
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NFL** using **2018** data.

In [4]:
def nfl_correlation():
    import pandas as pd
    import numpy as np
    import scipy.stats as stats
    import re

    cities=pd.read_html("assets/wikipedia_data.html")[1]
    cities=cities.iloc[:-1,[0,3,5,6,7,8]]
    cities.replace('\[\w.*\]','', regex=True, inplace=True) # reemplazo [] por espacio ''
    cities.replace('—',0, regex=True, inplace=True)         # reemplazo — por 0 = NaN
    cities.replace('',0, regex=True, inplace=True)          # reemplazo '' por 0 = NaN
    cities['NFL']=cities['NFL'].str.strip()
    copy_NFL=cities[['Metropolitan area','NFL','Population (2016 est.)[8]']].dropna()
    copy_NFL.sort_values(by=['NFL'], inplace=True)  #len=26
    copy_NFL['Population (2016 est.)[8]']=copy_NFL['Population (2016 est.)[8]'].astype('int64')

 
    population_by_region=copy_NFL['Population (2016 est.)[8]']


    nfl_df=pd.read_csv("assets/nfl.csv")
    nfl_df=nfl_df.iloc[:40,[1,2,11,13,14]] #cargo las columnas que necesito
    nfl_df=nfl_df.drop(nfl_df.index[[0,5,10,15,20,25,30,35]], axis=0)
    nfl_df['W']=nfl_df['W'].astype('int64')                  #convierto columna str a int para poder dividir
    nfl_df['L']=nfl_df['L'].astype('int64')                  #convierto columna str a int para poder dividir
    nfl_df['W/L Ratio']=nfl_df['W']/(nfl_df['W']+ nfl_df['L'])  #convierto columna str a int
    nfl_df.replace('\*','', regex=True, inplace=True)
    nfl_df.replace('\+','', regex=True, inplace=True)
    nfl_df['team'].replace("[\D].*\s",'',inplace=True,regex=True)
    nfl_df['team']=nfl_df['team'].str.strip()
    nfl_df['team'].replace({'Giants' : 'GiantsJets', 'Rams': 'RamsChargers', '49ers': '49ersRaiders'}, inplace=True)
    nfl_df.loc[24,('W/L Ratio')]=(nfl_df.loc[24,('W/L Ratio')] + nfl_df.loc[4,('W/L Ratio')])/2
    nfl_df.loc[36,('W/L Ratio')]=(nfl_df.loc[36,('W/L Ratio')] + nfl_df.loc[17,('W/L Ratio')])/2
    nfl_df.loc[38,('W/L Ratio')]=(nfl_df.loc[38,('W/L Ratio')] + nfl_df.loc[19,('W/L Ratio')])/2
    nfl_df=nfl_df.drop([4,17,19],axis=0)
    nfl_df.sort_values(by=['team'], inplace=True)

    win_loss_by_region = nfl_df['W/L Ratio']


    corr, pval= stats.pearsonr(population_by_region, win_loss_by_region)
    print(corr)
    return corr

nfl_correlation()

0.004922112149349429


0.004922112149349429

## Question 5
In this question I would like you to explore the hypothesis that **given that an area has two sports teams in different sports, those teams will perform the same within their respective sports**. How I would like to see this explored is with a series of paired t-tests (so use [`ttest_rel`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_rel.html)) between all pairs of sports. Are there any sports where we can reject the null hypothesis? Again, average values where a sport has multiple teams in one region. Remember, you will only be including, for each sport, cities which have teams engaged in that sport, drop others as appropriate. This question is worth 20% of the grade for this assignment.

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

mlb_df=pd.read_csv("assets/mlb.csv")
nhl_df=pd.read_csv("assets/nhl.csv")
nba_df=pd.read_csv("assets/nba.csv")
nfl_df=pd.read_csv("assets/nfl.csv")
cities=pd.read_html("assets/wikipedia_data.html")[1]
cities=cities.iloc[:-1,[0,3,5,6,7,8]]

def sports_team_performance():
    # YOUR CODE HERE
    raise NotImplementedError()
    
    # Note: p_values is a full dataframe, so df.loc["NFL","NBA"] should be the same as df.loc["NBA","NFL"] and
    # df.loc["NFL","NFL"] should return np.nan
    sports = ['NFL', 'NBA', 'NHL', 'MLB']
    p_values = pd.DataFrame({k:np.nan for k in sports}, index=sports)
    
    assert abs(p_values.loc["NBA", "NHL"] - 0.02) <= 1e-2, "The NBA-NHL p-value should be around 0.02"
    assert abs(p_values.loc["MLB", "NFL"] - 0.80) <= 1e-2, "The MLB-NFL p-value should be around 0.80"
    return p_values